In [1]:
import numpy as np
import pandas as pd
import mplfinance as mpf
import matplotlib
import os

matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from PIL import Image
from joblib import Parallel, delayed
from tqdm import tqdm
print("Finished importing Libraries")

Finished importing Libraries


In [ ]:

##############################
#       USER PARAMETERS      #
##############################

#CSV_PATH = r"CRSP_kun_3_aktier_test_data.csv"
CSV_PATH = os.path.abspath("crsp_top500_daily_1993_2003.csv")
#CSV_PATH = os.path.abspath("CRSP_kun_3_aktier_test_data.csv")
print("Current working directory:", os.getcwd())
print("Absolute CSV path:", CSV_PATH)
print("Exists:", os.path.exists(CSV_PATH))

df = pd.read_csv(CSV_PATH)

# -----------------------------
# Add or remove elements from the generated images here
# -----------------------------

##########################################
dataset_type = "train" # enten "test" eller "train" ####### VÆLG HER OG IKKE I IF STATEMENT ####################################
TICKER_LIMIT = None      # Set to an integer to process only a subset of tickers. SÆTTES TIL "NONE" VED UBEGRÆNSET
#########################################

TICKER_START_INDEX = 0    # Skip tickers before this index for chunked processing

COLOR_MODE = "color"      # "color" or "grayscale"
CHART_TYPE = "candlestick"       # "ohlc" or "candlestick"
USE_PRICE_CHART = True

WINDOW_SIZE = 5  # Number of days for the chart 
HORIZON = 5       # How far in the future we try to predict (lasdt day in windows + "Horizon" days ahead)

USE_MA = False

USE_BB = False         # <-- set True to compute BB columns
BB_NUM_STD = 2

USE_VOLUME = False

USE_RSI = True   # <-- set True to compute + overlay RSI
RSI_PERIOD = 14

USE_MACD = False  # Set True to compute and display MACD

MACD_FAST_PERIOD = 4 # standard 12
MACD_SLOW_PERIOD = 9 # standard 26
MACD_SIGNAL_PERIOD = 3 # standard 9





if dataset_type == "test":
    overlap = 5  # Enten "1" som gør at der er overlap eller "5" som gør at der ikke er overlap mellem data i billederne.
    START_DATE = pd.to_datetime("2001-01-01")
    END_DATE   = pd.to_datetime("2003-12-31") 
elif dataset_type == "train":
    START_DATE = pd.to_datetime("1993-01-04") 
    END_DATE   = pd.to_datetime("2000-12-31")
    overlap = 1

# Head folder for this exact image combination
# This folder will contain:
#   train/
#   test/
#   train_labels.csv
#   test_labels.csv

folder_name_parts = [COLOR_MODE]

if USE_PRICE_CHART:
    folder_name_parts.append(CHART_TYPE)

if USE_VOLUME:
    folder_name_parts.append("volume")

if USE_MA:
    folder_name_parts.append("ma")

if USE_BB:
    folder_name_parts.append("bb")

if USE_RSI:
    folder_name_parts.append("rsi_14_quarter_panel")

if USE_MACD:
    folder_name_parts.append("macd_4_9_3")

HEAD_DIR = "_".join(folder_name_parts)

# Images go into either the train or test subfolder
OUTPUT_DIR = os.path.join(HEAD_DIR, dataset_type)

# Labels go directly inside the head folder
LABELS_CSV_PATH = os.path.join(HEAD_DIR, f"{dataset_type}_labels.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("HEAD_DIR is:", HEAD_DIR)
print("OUTPUT_DIR is:", OUTPUT_DIR)
print("LABELS_CSV_PATH is:", LABELS_CSV_PATH)
print("OUTPUT_DIR exists?", os.path.exists(OUTPUT_DIR))




##############################
#       DATA PREPARATION     #
##############################


df['DlyCalDt'] = pd.to_datetime(df['DlyCalDt'], format='%Y-%m-%d')
#df = df.sort_values(by='DlyCalDt')
df.rename(columns={
    'PERMNO': 'Ticker',
    'DlyOpen': 'Open',
    'DlyHigh': 'High',
    'DlyLow': 'Low',
    'DlyClose': 'Close',
    'DlyVol': 'Volume',
    'DlyCalDt': 'Date'
}, inplace=True)

# Filter data to the desired date range
df = df[(df['Date'] >= START_DATE) & (df['Date'] <= END_DATE)].copy()

# Group the data by ticker
grouped = list(df.groupby('Ticker'))

# --- Determine warm-up (how many initial rows to skip) ---
warmup = 0


# -----------------------------
# MACD toggle + per-ticker compute
# -----------------------------

def _compute_macd(
    close: pd.Series,
    fast_period: int = 12,
    slow_period: int = 26,
    signal_period: int = 9
):
    """
    Calculate MACD, signal line and MACD histogram.

    MACD line:
        EMA_fast - EMA_slow

    Signal line:
        EMA of the MACD line

    Histogram:
        MACD line - signal line
    """

    if fast_period <= 0 or slow_period <= 0 or signal_period <= 0:
        raise ValueError("All MACD periods must be positive integers.")

    if fast_period >= slow_period:
        raise ValueError(
            "MACD_FAST_PERIOD must be smaller than MACD_SLOW_PERIOD."
        )

    close = pd.to_numeric(close, errors="coerce").astype(float)

    ema_fast = close.ewm(
        span=fast_period,
        adjust=False,
        min_periods=fast_period,
        ignore_na=True
    ).mean()

    ema_slow = close.ewm(
        span=slow_period,
        adjust=False,
        min_periods=slow_period,
        ignore_na=True
    ).mean()

    macd_line = ema_fast - ema_slow

    signal_line = macd_line.ewm(
        span=signal_period,
        adjust=False,
        min_periods=signal_period,
        ignore_na=True
    ).mean()

    histogram = macd_line - signal_line

    macd_line.name = "MACD_LINE"
    signal_line.name = "MACD_SIGNAL"
    histogram.name = "MACD_HIST"

    return macd_line, signal_line, histogram


if USE_MACD:
    grouped_with_macd = []

    for tkr, g in grouped:
        g = g.sort_values("Date").copy()

        macd_line, signal_line, histogram = _compute_macd(
            close=g["Close"],
            fast_period=MACD_FAST_PERIOD,
            slow_period=MACD_SLOW_PERIOD,
            signal_period=MACD_SIGNAL_PERIOD
        )

        g["MACD_LINE"] = macd_line
        g["MACD_SIGNAL"] = signal_line
        g["MACD_HIST"] = histogram

        grouped_with_macd.append((tkr, g))

    grouped = grouped_with_macd


if USE_MACD:
    # First valid signal line and histogram occur at:
    # slow_period + signal_period - 2
    macd_warmup = MACD_SLOW_PERIOD + MACD_SIGNAL_PERIOD - 2
    warmup = max(warmup, macd_warmup)

# -----------------------------
# END OF: MACD
# -----------------------------


# -----------------------------
# RSI toggle + per-ticker compute (no cross-ticker leakage)
# -----------------------------

def _compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """
    Calculate the standard Wilder RSI.

    The first valid RSI occurs at positional index `period`,
    because `period` price changes require `period + 1` closing prices.
    """
    close = pd.to_numeric(close, errors="coerce").astype(float)

    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = pd.Series(np.nan, index=close.index, dtype=float)
    avg_loss = pd.Series(np.nan, index=close.index, dtype=float)

    # Not enough closing prices to calculate the first RSI
    if len(close) <= period:
        return pd.Series(np.nan, index=close.index, name="RSI")

    # Initial Wilder averages based on the first `period` changes
    avg_gain.iloc[period] = gain.iloc[1 : period + 1].mean()
    avg_loss.iloc[period] = loss.iloc[1 : period + 1].mean()

    # Wilder recursive smoothing
    for i in range(period + 1, len(close)):
        avg_gain.iloc[i] = (
            avg_gain.iloc[i - 1] * (period - 1) + gain.iloc[i]
        ) / period

        avg_loss.iloc[i] = (
            avg_loss.iloc[i - 1] * (period - 1) + loss.iloc[i]
        ) / period

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    # Special cases
    rsi = rsi.mask((avg_loss == 0) & (avg_gain > 0), 100.0)
    rsi = rsi.mask((avg_gain == 0) & (avg_loss > 0), 0.0)
    rsi = rsi.mask((avg_gain == 0) & (avg_loss == 0), 50.0)

    rsi.name = "RSI"
    return rsi

if USE_RSI:
    # Build a new grouped list where each ticker DataFrame includes an RSI column
    grouped_with_rsi = []
    for tkr, g in grouped:
        g = g.sort_values('Date')
        g['RSI'] = _compute_rsi(g['Close'], RSI_PERIOD)
        grouped_with_rsi.append((tkr, g))
    grouped = grouped_with_rsi

if USE_RSI:
    warmup = max(warmup, RSI_PERIOD)   # safe default
# -----------------------------
# END of RSI
# -----------------------------

# -----------------------------
# Bollinger Bands toggle + per-ticker compute (no cross-ticker leakage)
# -----------------------------


def _compute_bollinger_bands(close: pd.Series, period: int, num_std: float):
    ma = close.rolling(window=period, min_periods=period).mean()
    sd = close.rolling(window=period, min_periods=period).std(ddof=0)  # ddof=0 is common in BB implementations
    upper = ma + num_std * sd
    lower = ma - num_std * sd
    return ma, upper, lower

if USE_BB:
    grouped_with_bb = []
    for tkr, g in grouped:
        g = g.sort_values('Date').copy()

        bb_mid, bb_upper, bb_lower = _compute_bollinger_bands(
            g['Close'], period=WINDOW_SIZE, num_std=BB_NUM_STD
        )

        g['BB_MID'] = bb_mid
        g['BB_UPPER'] = bb_upper
        g['BB_LOWER'] = bb_lower

        grouped_with_bb.append((tkr, g))

    grouped = grouped_with_bb

# Bollinger Bands warmup: first BB_PERIOD-1 rows are NaN
if USE_BB:
    warmup = max(warmup, WINDOW_SIZE - 1)

# -----------------------------
# END OF: Bollinger Bands
# -----------------------------

# -----------------------------
# Moving Average toggle + parameters
# -----------------------------

def _compute_moving_average(close: pd.Series, period: int) -> pd.Series:
    return close.rolling(window=period, min_periods=period).mean()

if USE_MA:
    grouped_with_ma = []
    for tkr, g in grouped:
        g = g.sort_values('Date').copy()

        g['MA'] = _compute_moving_average(g['Close'], WINDOW_SIZE)

        grouped_with_ma.append((tkr, g))

    grouped = grouped_with_ma

# Moving Average warmup: first MA_PERIOD-1 rows are NaN
if USE_MA:
    warmup = max(warmup, WINDOW_SIZE - 1)
# -----------------------------
# END OF: Moving Average
# -----------------------------


warmup = 33

print("Warmup: ", warmup)


if TICKER_LIMIT is not None:
    grouped = grouped[:TICKER_LIMIT]
# Apply ticker start index to allow chunked processing
grouped = grouped[TICKER_START_INDEX:]
print(f"Processing {len(grouped)} tickers from {START_DATE.date()} to {END_DATE.date()}...")

##############################
#   HELPER FUNCTIONALITY     #
##############################

def create_mplfinance_chart(
    data_window,
    save_path,
    mav,
    chart_type,
    color_mode,
    xlim=None,
    add_price=True,
    add_rsi=False,
    add_macd=False,
    add_bb=False,
    add_ma=False,
    add_volume=False
):
    """
    Create a financial chart using mplfinance.

    If add_price=True:
        - OHLC/candlesticks are visible in panel 0.
        - MA and Bollinger Bands are drawn on panel 0.
        - Volume, RSI and MACD receive separate panels.

    If add_price=False:
        - OHLC/candlesticks are still technically generated, but all
          price-chart colors are black and therefore invisible.
        - Panel 0 receives an almost-zero panel ratio.
        - MA/BB, volume, RSI and MACD are drawn in secondary panels.
        - A single active indicator therefore occupies almost the
          entire image.
    """

    # --------------------------------------------------
# MACD display choice
# --------------------------------------------------
# Available choices:
#   "lines"     = MACD line + signal line
#   "histogram" = MACD histogram only
#   "full"      = MACD line + signal line + histogram

    MACD_DISPLAY = "lines"

    valid_macd_displays = {
        "lines",
        "histogram",
        "full"
    }

    if MACD_DISPLAY not in valid_macd_displays:
        raise ValueError(
            "MACD_DISPLAY must be 'lines', 'histogram' or 'full'."
        )

    # --------------------------------------------------
    # Validate chart type and define line widths
    # --------------------------------------------------

    if chart_type == "ohlc":
        width_config = {
            "ohlc_linewidth": 5.0,
            "ohlc_ticksize": 0.25,
            "volume_width": 0.8,
            "volume_linewidth": 0.0
        }

    elif chart_type == "candlestick":
        width_config = {
            "candle_width": 0.7,
            "candle_linewidth": 6.0,
            "volume_width": 0.8,
            "volume_linewidth": 0.0
        }

    else:
        raise ValueError(
            "CHART_TYPE must be either 'ohlc' or 'candlestick'."
        )

    # --------------------------------------------------
    # Validate columns
    # --------------------------------------------------

    required_price_columns = {
        "Open",
        "High",
        "Low",
        "Close"
    }

    missing_price_columns = (
        required_price_columns - set(data_window.columns)
    )

    if missing_price_columns:
        raise KeyError(
            f"Missing required price columns: "
            f"{missing_price_columns}"
        )

    if add_ma and "MA" not in data_window.columns:
        raise KeyError("MA column is missing from data_window.")

    if add_bb:
        required_bb_columns = {
            "BB_UPPER",
            "BB_MID",
            "BB_LOWER"
        }

        missing_bb_columns = (
            required_bb_columns - set(data_window.columns)
        )

        if missing_bb_columns:
            raise KeyError(
                f"Missing Bollinger Band columns: "
                f"{missing_bb_columns}"
            )

    if add_volume and "Volume" not in data_window.columns:
        raise KeyError(
            "Volume column is missing from data_window."
        )

    if add_rsi and "RSI" not in data_window.columns:
        raise KeyError(
            "RSI column is missing from data_window."
        )

    if add_macd:

        if MACD_DISPLAY == "lines":
            required_macd_columns = {
                "MACD_LINE",
                "MACD_SIGNAL"
            }

        elif MACD_DISPLAY == "histogram":
            required_macd_columns = {
                "MACD_HIST"
            }

        else:  # MACD_DISPLAY == "full"
            required_macd_columns = {
                "MACD_LINE",
                "MACD_SIGNAL",
                "MACD_HIST"
            }

        missing_macd_columns = (
            required_macd_columns - set(data_window.columns)
        )

        if missing_macd_columns:
            raise KeyError(
                f"Missing MACD columns for "
                f"MACD_DISPLAY='{MACD_DISPLAY}': "
                f"{missing_macd_columns}"
            )
    

    # --------------------------------------------------
    # Indicator colors
    # --------------------------------------------------

    if color_mode == "grayscale":

        visible_up_color = "1.0"
        visible_down_color = "1.0" # "0.25"

        volume_color = "1.0" # "0.55"

        ma_color = "1.0" # "0.70"

        bb_upper_color = "0.75"
        bb_mid_color = "0.50"
        bb_lower_color = "0.75"

        rsi_color = "0.75"

        macd_line_color = "1.0"
        macd_signal_color = "0.65"
        macd_hist_color = "0.40"
        zero_line_color = "0.35"
         

    elif color_mode == "color":

        visible_up_color = "black" # default green ellers black for reducering af panel 
        visible_down_color = "black" # default red ellers black for reducering af panel 

        volume_color = "blue"

        ma_color = "yellow"

        bb_upper_color = "yellow"
        bb_mid_color = "orange"
        bb_lower_color = "yellow"

        rsi_color = "orange"

        macd_line_color = "blue"
        macd_signal_color = "orange"
        macd_hist_color = "gray" ################# Sæt til sort hvis den skal væl
        zero_line_color = "lime"

    else:
        raise ValueError(
            "COLOR_MODE must be either "
            "'color' or 'grayscale'."
        )

    # --------------------------------------------------
    # Hide or display OHLC/candlesticks
    # --------------------------------------------------

    if add_price:
        up_color = visible_up_color
        down_color = visible_down_color

        edge_colors = {
            "up": visible_up_color,
            "down": visible_down_color
        }

        wick_colors = {
            "up": visible_up_color,
            "down": visible_down_color
        }

        ohlc_colors = {
            "up": visible_up_color,
            "down": visible_down_color
        }

    else:
        # The price chart is still generated, but every component
        # has the same black color as the background.
        up_color = "black"
        down_color = "black"

        edge_colors = {
            "up": "black",
            "down": "black"
        }

        wick_colors = {
            "up": "black",
            "down": "black"
        }

        ohlc_colors = {
            "up": "black",
            "down": "black"
        }

    market_colors = mpf.make_marketcolors(
        up=up_color,
        down=down_color,
        edge=edge_colors,
        wick=wick_colors,
        ohlc=ohlc_colors,

        # Built-in volume is not used below, but this keeps
        # the market-color definition complete.
        volume=volume_color
    )

    # Explicitly make both the figure and every panel black.
    plot_style = mpf.make_mpf_style(
        base_mpf_style="charles",
        marketcolors=market_colors,
        facecolor="black",
        figcolor="black",
        gridcolor="black",
        y_on_right=False
    )

    # --------------------------------------------------
    # Construct panels dynamically
    # --------------------------------------------------

    addplots = []

    # mplfinance always reserves panel 0 for the price chart.
    #
    # When price is hidden, its panel is made extremely small.
    # It cannot be exactly zero, because matplotlib requires
    # panel ratios to be positive.
    if add_price:
        panel_ratios = [3.0]
    else:
        panel_ratios = [0.01]

    next_panel = 1

    # --------------------------------------------------
    # MA and Bollinger Bands (overlap med Candlesticks/OHLC i samme panel)
    # --------------------------------------------------

    # if add_price:
    #     # When the price chart is visible, MA and BB are overlays
    #     # on the main price panel.
    #     price_indicator_panel = 0

    # elif add_ma or add_bb:
    #     # When the price chart is hidden, MA and BB get their
    #     # own large secondary panel.
    #     price_indicator_panel = next_panel
    #     panel_ratios.append(1.0)
    #     next_panel += 1

    # else:
    #     price_indicator_panel = None

    # --------------------------------------------------
    # MA and Bollinger Bands (MA og BB kommer i et seperat panel)
    # --------------------------------------------------

    if add_ma or add_bb:
        # MA and BB always receive their own panel,
        # separate from the candlestick/OHLC price panel.
        price_indicator_panel = next_panel

        # Relative height of the MA + BB panel
        panel_ratios.append(3.0)

        next_panel += 1

    else:
        price_indicator_panel = None

    if add_bb:
        addplots.extend([
            mpf.make_addplot(
                data_window["BB_UPPER"],
                panel=price_indicator_panel,
                color=bb_upper_color,
                secondary_y=False
            ),

            mpf.make_addplot(
                data_window["BB_MID"],
                panel=price_indicator_panel,
                color=bb_mid_color,
                secondary_y=False
            ),

            mpf.make_addplot(
                data_window["BB_LOWER"],
                panel=price_indicator_panel,
                color=bb_lower_color,
                secondary_y=False
            )
        ])

    if add_ma:
        addplots.append(
            mpf.make_addplot(
                data_window["MA"],
                panel=price_indicator_panel,
                color=ma_color,
                secondary_y=False
            )
        )

    # --------------------------------------------------
    # Volume
    # --------------------------------------------------

    if add_volume:
        volume_panel = next_panel
        panel_ratios.append(1.0)
        next_panel += 1

        # Volume is deliberately created as an addplot rather
        # than with volume=True. This gives full control over
        # its panel number and panel ratio.
        addplots.append(
            mpf.make_addplot(
                data_window["Volume"],
                panel=volume_panel,
                type="bar",
                color=volume_color,
                secondary_y=False
            )
        )

    # --------------------------------------------------
    # RSI
    # --------------------------------------------------

    if add_rsi:
        rsi_panel = next_panel
        panel_ratios.append(1.0) # default 2
        next_panel += 1

        addplots.append(
            mpf.make_addplot(
                data_window["RSI"],
                panel=rsi_panel,
                color=rsi_color,
                ylim=(0, 100),
                secondary_y=False
            )
        )

    # --------------------------------------------------
    # MACD
    # --------------------------------------------------

    if add_macd:
        macd_panel = next_panel
        panel_ratios.append(3.0)
        next_panel += 1

        macd_addplots = []

        # Horizontal zero baseline
        macd_zero_line = pd.Series(
            0.0,
            index=data_window.index
        )

        macd_addplots.append(
            mpf.make_addplot(
                macd_zero_line,
                panel=macd_panel,
                color=zero_line_color,
                width=0.8,
                secondary_y=False
            )
        )

        # Histogram is added first, so lines are drawn on top
        # when MACD_DISPLAY == "full".
        if MACD_DISPLAY in {"histogram", "full"}:
            macd_addplots.append(
                mpf.make_addplot(
                    data_window["MACD_HIST"],
                    panel=macd_panel,
                    type="bar",
                    color=macd_hist_color,
                    secondary_y=False
                )
            )

        if MACD_DISPLAY in {"lines", "full"}:
            macd_addplots.extend([
                mpf.make_addplot(
                    data_window["MACD_LINE"],
                    panel=macd_panel,
                    color=macd_line_color,
                    secondary_y=False
                ),

                mpf.make_addplot(
                    data_window["MACD_SIGNAL"],
                    panel=macd_panel,
                    color=macd_signal_color,
                    secondary_y=False
                )
            ])

        addplots.extend(macd_addplots)

    # --------------------------------------------------
    # Check that indicator-only images have content
    # --------------------------------------------------

    if (
        not add_price
        and not add_ma
        and not add_bb
        and not add_volume
        and not add_rsi
        and not add_macd
    ):
        raise ValueError(
            "The price chart is hidden, but no indicator has "
            "been selected. Activate at least one of MA, BB, "
            "volume, RSI or MACD."
        )

    # --------------------------------------------------
    # Plot arguments
    # --------------------------------------------------

    resolved_xlim = (
        xlim
        if xlim is not None
        else (-0.5, len(data_window) - 0.5)
    )

    plot_arguments = {
        "type": chart_type,

        # Must remain False because volume is handled manually
        # as an addplot above.
        "volume": False,

        "style": plot_style,
        "axisoff": True,
        "returnfig": True,
        "figsize": (4, 4),
        "update_width_config": width_config,
        "panel_ratios": tuple(panel_ratios),
        "xlim": resolved_xlim
    }

    if addplots:
        plot_arguments["addplot"] = addplots

    # --------------------------------------------------
    # Generate figure
    # --------------------------------------------------

    fig, axes = mpf.plot(
        data_window,
        **plot_arguments
    )

    # mplfinance can return both primary and secondary axes.
    # Explicitly setting all of them to black avoids white
    # panel backgrounds from the base style.
    fig.patch.set_facecolor("black")

    for axis in axes:
        axis.set_facecolor("black")
        axis.set_axis_off()

    # --------------------------------------------------
    # Save image
    # --------------------------------------------------

    fig.savefig(
        save_path,
        bbox_inches="tight",
        pad_inches=0,
        facecolor="black"
    )

    plt.close(fig)

    # --------------------------------------------------
    # Resize to the fixed CNN resolution
    # --------------------------------------------------

    with Image.open(save_path) as image:
        resized_image = image.resize(
            (15, 32),
            Image.Resampling.BILINEAR
        ).convert("RGB")

        resized_image.save(save_path)

def generate_images_for_ticker(ticker, ticker_data, window_size, horizon, output_dir):
    """
    "Horizon" is the variable that decides how far in the future the code try to predict.
    To compute the moving average correctly over the window period, extend the data backward by
    (window_size - 1) rows (if available). Images are saved in a subfolder under output_dir,
    and filenames include the ticker and the date range.
    """

    # Create a subfolder for the ticker.
    ticker_folder = os.path.join(output_dir, str(ticker)) # default
    # if os.path.exists(ticker_folder):
    #     print(f"Skipping {ticker}: folder already exists")
    #     return []
    os.makedirs(ticker_folder, exist_ok=True) # default


    ticker_data = ticker_data.copy()

    if "Date" in ticker_data.columns:
        ticker_data = ticker_data.sort_values("Date").set_index("Date")
    else:
        ticker_data = ticker_data.sort_index()

    ticker_data["future_close"] = ticker_data["Close"].shift(-horizon)
    ticker_data = ticker_data.dropna(subset=["future_close"]).copy()
    ticker_data["label"] = (ticker_data["future_close"] > ticker_data["Close"]).astype(int)

    n = len(ticker_data)
    label_rows = []
    


    #Nedestående burde slette tidligere billeder, så du er sikker på dem der kommer nu er nye
    for f in os.listdir(ticker_folder):
        if f.endswith(".png"):
            os.remove(os.path.join(ticker_folder, f))
    
    window_index = 0
    # Loop using non-overlapping windows with step = window_size
    for i in range(warmup, n - window_size + 1, overlap):
        # Get the current window for display.
        data_to_plot = ticker_data.iloc[i : i + window_size]
        anchor_row = data_to_plot.iloc[-1]
        close_now = anchor_row["Close"]
        close_future = anchor_row["future_close"]
        label = anchor_row["label"]
        
        start_str = data_to_plot.index[0].strftime("%Y-%m-%d")
        end_str = data_to_plot.index[-1].strftime("%Y-%m-%d")
        filename = f"{ticker}_{start_str}_to_{end_str}_{window_index}.png"
        filepath = os.path.join(ticker_folder, filename)

        
        try:
            create_mplfinance_chart(
                data_to_plot, 
                filepath, 
                mav=window_size,
                #resolution=RESOLUTION,
                chart_type=CHART_TYPE,
                color_mode=COLOR_MODE, 
                xlim=None, 
                add_price=USE_PRICE_CHART,
                add_rsi=USE_RSI,                 
                add_bb=USE_BB, 
                add_ma=USE_MA, 
                add_volume=USE_VOLUME, 
                add_macd=USE_MACD,
                )
            label_rows.append({
                "image_path": filepath,
                "ticker": ticker,
                "start_date": start_str,
                "end_date": end_str,
                "close_now": close_now,
                "close_future": close_future,
                "label": label
            })
        except Exception as e:
            print(f"Error processing ticker {ticker} window {window_index}: {e}")
        window_index += 1

    return label_rows

def process_ticker_wrapper(ticker_tuple):
    ticker, ticker_data = ticker_tuple
    label_rows = generate_images_for_ticker(ticker, ticker_data.copy(), WINDOW_SIZE, HORIZON, OUTPUT_DIR)
    return label_rows

##############################
#       RUN PIPELINE         #
##############################

def run_pipeline_joblib(tickers_group, n_jobs=-1):   
    """
    Process all tickers in parallel using joblib. Setting n_jobs=-1 utilizes all available CPU cores.
    """
    results = Parallel(n_jobs=n_jobs, prefer="processes")( # sæt (prefer="threads") hvis du vil have print("") virker så du kan se hvad der sker. Brug (prefer="processes") for at koden eksekvere hurtigere.
        delayed(process_ticker_wrapper)(ticker_tuple) for ticker_tuple in tqdm(tickers_group, desc="Processing tickers")
    )

    all_label_rows = []
    for ticker_rows in results:
        all_label_rows.extend(ticker_rows)

    labels_df = pd.DataFrame(all_label_rows)
    labels_df.to_csv(LABELS_CSV_PATH, index=False)

    print(f"Saved labels CSV to: {LABELS_CSV_PATH}")
    print("Image generation complete!")

# Run the pipeline using joblib with all available cores
print("Done initial data loading. Now Starting to create images")
run_pipeline_joblib(grouped, n_jobs=-1)

Current working directory: c:\Users\morte\Desktop\speciale-i-datavidenskab-2026
Absolute CSV path: c:\Users\morte\Desktop\speciale-i-datavidenskab-2026\crsp_top500_daily_1993_2003.csv
Exists: True
HEAD_DIR is: color_candlestick_rsi_14_quarter_panel
OUTPUT_DIR is: color_candlestick_rsi_14_quarter_panel\train
LABELS_CSV_PATH is: color_candlestick_rsi_14_quarter_panel\train_labels.csv
OUTPUT_DIR exists? True
Warmup:  15
Processing 1227 tickers from 1993-01-04 to 2000-12-31...
Done initial data loading. Now Starting to create images


Processing tickers:   4%|▍         | 48/1227 [01:17<31:36,  1.61s/it]c:\Users\morte\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Processing tickers: 100%|██████████| 1227/1227 [1:14:48<00:00,  3.66s/it]


Saved labels CSV to: color_candlestick_rsi_14_quarter_panel\train_labels.csv
Image generation complete!
